# Polymarket Whale Monitor

This notebook monitors Polymarket for **"Fresh Whale"** activity - detecting when newly created accounts suddenly invest large sums (>$10,000 USD) into prediction markets.

## Features
- Real-time monitoring via Polymarket APIs
- Detects large trades from new/inactive accounts
- Discord webhook notifications
- Configurable thresholds

## Data Sources
- **Activity Subgraph**: Large trade detection (splits)
- **Gamma API**: Market information lookup
- **Data API**: User trading history

## Setup Instructions
1. Run the installation cell below
2. Configure your Discord webhook URL
3. Adjust thresholds as needed
4. Run the monitor!

---

## Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install -q requests pandas python-dateutil

print("Dependencies installed successfully!")

## Step 2: Configuration

### How to Get a Discord Webhook URL:

1. Open Discord and go to your server
2. Right-click on the channel where you want alerts
3. Select **"Edit Channel"**
4. Go to **"Integrations"** in the left sidebar
5. Click **"Webhooks"**
6. Click **"New Webhook"**
7. Give it a name (e.g., "Whale Monitor")
8. Click **"Copy Webhook URL"**
9. Paste it below!

---

In [ ]:
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

# Your Discord Webhook URL (REQUIRED for alerts)
# Leave empty to just log locally without Discord notifications
DISCORD_WEBHOOK_URL = ""  # Paste your webhook URL here

# Minimum trade value to trigger an alert (in USD)
MIN_TRADE_VALUE_USD = 10000  # $10,000 default

# "New Account" Detection Criteria:
# An account is considered "fresh" if EITHER:
#   - It has fewer than MAX_HISTORICAL_TRADES total trades, OR
#   - Its first trade was within the last NEW_ACCOUNT_HOURS hours

MAX_HISTORICAL_TRADES = 5  # Max trades to be considered "new"
NEW_ACCOUNT_HOURS = 72     # Account age threshold (72 hours = 3 days)

# Polling Configuration
POLL_INTERVAL_SECONDS = 60  # Check for new trades every 60 seconds
LOOKBACK_MINUTES = 5        # Look back 5 minutes each poll

print("Configuration loaded!")
print(f"  - Minimum trade value: ${MIN_TRADE_VALUE_USD:,}")
print(f"  - Max historical trades: {MAX_HISTORICAL_TRADES}")
print(f"  - New account threshold: {NEW_ACCOUNT_HOURS} hours")
print(f"  - Poll interval: {POLL_INTERVAL_SECONDS} seconds")
if DISCORD_WEBHOOK_URL:
    print("  - Discord webhook: Configured")
else:
    print("  - Discord webhook: NOT SET (alerts will only be logged)")

## Step 3: Core Monitoring Code

Run this cell to load all the monitoring functions.

In [ ]:
import os
import sys
import time
import json
import logging
from datetime import datetime, timedelta, timezone
from typing import Optional, Dict, List, Any, Set
from dataclasses import dataclass, field
from IPython.display import display, HTML, clear_output

import requests
import pandas as pd

# =============================================================================
# CONFIGURATION CLASS
# =============================================================================

@dataclass
class Config:
    """Configuration settings for the Whale Monitor."""
    # API Endpoints
    ACTIVITY_SUBGRAPH_URL: str = (
        "https://api.goldsky.com/api/public/"
        "project_cl6mb8i9h0003e201j6li0diw/subgraphs/activity-subgraph/0.0.4/gn"
    )
    GAMMA_API_URL: str = "https://gamma-api.polymarket.com"
    DATA_API_URL: str = "https://data-api.polymarket.com"
    
    DISCORD_WEBHOOK_URL: str = ""
    MIN_TRADE_VALUE_USD: float = 10_000.0
    MAX_HISTORICAL_TRADES: int = 5
    NEW_ACCOUNT_HOURS: int = 72
    POLL_INTERVAL_SECONDS: int = 60
    LOOKBACK_MINUTES: int = 5
    REQUEST_TIMEOUT: int = 30
    MAX_RETRIES: int = 3
    RETRY_DELAY: int = 5
    RATE_LIMIT_DELAY: float = 0.2
    LOG_LEVEL: str = "INFO"

# =============================================================================
# DATA MODELS
# =============================================================================

@dataclass
class Trade:
    """Represents a trading event on Polymarket."""
    id: str
    user_address: str
    market_id: str
    market_title: str
    outcome: str
    amount: float
    price: float
    value_usd: float
    timestamp: int
    tx_hash: str

    @property
    def formatted_time(self) -> str:
        return datetime.fromtimestamp(
            self.timestamp, tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")


@dataclass
class AccountProfile:
    """Profile information for a Polymarket account."""
    address: str
    total_trades: int
    first_trade_timestamp: Optional[int]
    total_volume_usd: float
    markets_traded: int

    @property
    def account_age_hours(self) -> Optional[float]:
        if not self.first_trade_timestamp:
            return None
        age_seconds = time.time() - self.first_trade_timestamp
        return age_seconds / 3600

    def is_fresh_whale(self, config: Config) -> bool:
        if self.total_trades < config.MAX_HISTORICAL_TRADES:
            return True
        if self.account_age_hours is not None:
            if self.account_age_hours < config.NEW_ACCOUNT_HOURS:
                return True
        return False


@dataclass
class FreshWhaleAlert:
    """Alert data for a Fresh Whale detection."""
    trade: Trade
    profile: AccountProfile
    detection_reason: str
    alert_time: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

    def to_discord_embed(self) -> Dict[str, Any]:
        if self.trade.value_usd >= 100_000:
            color = 0xFF0000
        elif self.trade.value_usd >= 50_000:
            color = 0xFFA500
        else:
            color = 0x00FF00

        profile_url = f"https://polymarket.com/profile/{self.trade.user_address}"

        if self.profile.account_age_hours is not None:
            if self.profile.account_age_hours < 1:
                age_str = f"{int(self.profile.account_age_hours * 60)} minutes"
            elif self.profile.account_age_hours < 24:
                age_str = f"{self.profile.account_age_hours:.1f} hours"
            else:
                age_str = f"{self.profile.account_age_hours / 24:.1f} days"
        else:
            age_str = "Unknown"

        return {
            "title": "Fresh Whale Detected!",
            "description": "A new account just made a large trade on Polymarket.",
            "color": color,
            "fields": [
                {"name": "Market", "value": self.trade.market_title[:256], "inline": False},
                {"name": "Position", "value": f"**{self.trade.outcome}** @ ${self.trade.price:.3f}", "inline": True},
                {"name": "Amount Invested", "value": f"**${self.trade.value_usd:,.2f}**", "inline": True},
                {"name": "Shares Purchased", "value": f"{self.trade.amount:,.2f}", "inline": True},
                {"name": "Account Age", "value": age_str, "inline": True},
                {"name": "Total Prior Trades", "value": str(self.profile.total_trades), "inline": True},
                {"name": "Detection Reason", "value": self.detection_reason, "inline": True},
                {"name": "Wallet Address", "value": f"`{self.trade.user_address[:10]}...{self.trade.user_address[-8:]}`", "inline": False}
            ],
            "timestamp": self.alert_time.isoformat(),
            "footer": {"text": "Polymarket Whale Monitor"},
            "url": profile_url
        }

# =============================================================================
# POLYMARKET API CLIENT
# =============================================================================

class PolymarketClient:
    """Client for interacting with Polymarket APIs."""

    def __init__(self, config: Config):
        self.config = config
        self.session = requests.Session()
        self._market_cache: Dict[str, Dict] = {}

    def _request_with_retry(self, method: str, url: str, **kwargs) -> Optional[requests.Response]:
        kwargs.setdefault("timeout", self.config.REQUEST_TIMEOUT)
        for attempt in range(self.config.MAX_RETRIES):
            try:
                response = self.session.request(method, url, **kwargs)
                response.raise_for_status()
                return response
            except requests.exceptions.RequestException as e:
                print(f"Request failed (attempt {attempt + 1}): {e}")
                if attempt < self.config.MAX_RETRIES - 1:
                    time.sleep(self.config.RETRY_DELAY)
        return None

    def _graphql_query(self, query: str, variables: Optional[Dict] = None) -> Optional[Dict]:
        payload = {"query": query}
        if variables:
            payload["variables"] = variables

        response = self._request_with_retry(
            "POST",
            self.config.ACTIVITY_SUBGRAPH_URL,
            json=payload,
            headers={"Content-Type": "application/json"}
        )

        if not response:
            return None

        result = response.json()
        if "errors" in result:
            print(f"GraphQL errors: {result['errors']}")
            return None

        return result.get("data")

    def get_large_splits(self, since_timestamp: int, min_amount_raw: int, first: int = 100) -> List[Dict]:
        """Fetch large splits (trades) from Activity Subgraph."""
        query = """
        query GetLargeSplits($since: BigInt!, $minAmount: BigInt!, $first: Int!) {
            splits(
                first: $first,
                orderBy: timestamp,
                orderDirection: desc,
                where: {
                    timestamp_gte: $since,
                    amount_gte: $minAmount
                }
            ) {
                id
                timestamp
                stakeholder
                condition
                amount
            }
        }
        """

        variables = {
            "since": str(since_timestamp),
            "minAmount": str(min_amount_raw),
            "first": first
        }

        data = self._graphql_query(query, variables)
        if data and "splits" in data:
            return data["splits"]
        return []

    def get_market_info(self, condition_id: str) -> Optional[Dict]:
        """Get market information from Gamma API by condition ID."""
        if condition_id in self._market_cache:
            return self._market_cache[condition_id]

        time.sleep(self.config.RATE_LIMIT_DELAY)

        url = f"{self.config.GAMMA_API_URL}/markets"
        params = {"conditionId": condition_id}

        response = self._request_with_retry("GET", url, params=params)

        if response and response.status_code == 200:
            markets = response.json()
            if markets and len(markets) > 0:
                market = markets[0]
                self._market_cache[condition_id] = market
                return market

        return None

    def get_user_activity(self, user_address: str, limit: int = 100) -> List[Dict]:
        """Get user's trading activity from Data API."""
        time.sleep(self.config.RATE_LIMIT_DELAY)

        url = f"{self.config.DATA_API_URL}/activity"
        params = {"user": user_address.lower(), "limit": limit}

        response = self._request_with_retry("GET", url, params=params)

        if response and response.status_code == 200:
            return response.json()
        return []

    def get_account_profile(self, address: str) -> Optional[AccountProfile]:
        """Build account profile from user's trading history."""
        activities = self.get_user_activity(address, limit=1000)

        if not activities:
            return AccountProfile(
                address=address,
                total_trades=0,
                first_trade_timestamp=None,
                total_volume_usd=0,
                markets_traded=0
            )

        trades = [a for a in activities if a.get("type") == "TRADE"]

        if not trades:
            return AccountProfile(
                address=address,
                total_trades=0,
                first_trade_timestamp=None,
                total_volume_usd=0,
                markets_traded=0
            )

        trades_sorted = sorted(trades, key=lambda x: x.get("timestamp", 0))

        total_trades = len(trades)
        first_trade_ts = trades_sorted[0].get("timestamp") if trades_sorted else None
        total_volume = sum(t.get("usdcSize", 0) for t in trades)
        unique_markets = set(t.get("conditionId", "") for t in trades)

        return AccountProfile(
            address=address,
            total_trades=total_trades,
            first_trade_timestamp=first_trade_ts,
            total_volume_usd=total_volume,
            markets_traded=len(unique_markets)
        )

    def get_recent_trades(self, since_timestamp: int, min_value_usd: float = 0, first: int = 100) -> List[Trade]:
        """Fetch recent large trades."""
        min_amount_raw = int(min_value_usd * 1_000_000)
        splits = self.get_large_splits(since_timestamp, min_amount_raw, first)

        print(f"Found {len(splits)} large splits from subgraph")

        trades = []
        for split in splits:
            try:
                split_id = split["id"]
                timestamp = int(split["timestamp"])
                stakeholder = split["stakeholder"]
                condition = split["condition"]
                amount_raw = int(split["amount"])
                value_usd = amount_raw / 1_000_000

                market_info = self.get_market_info(condition)

                if market_info:
                    market_title = market_info.get("question", "Unknown Market")
                    market_id = market_info.get("id", condition)
                else:
                    market_title = f"Market {condition[:16]}..."
                    market_id = condition

                trade = Trade(
                    id=split_id,
                    user_address=stakeholder,
                    market_id=market_id,
                    market_title=market_title,
                    outcome="Position",
                    amount=amount_raw / 1_000_000,
                    price=1.0,
                    value_usd=value_usd,
                    timestamp=timestamp,
                    tx_hash=split_id.split("_")[0] if "_" in split_id else split_id
                )
                trades.append(trade)

            except (KeyError, ValueError, TypeError) as e:
                print(f"Failed to parse split: {e}")
                continue

        return trades

# =============================================================================
# DISCORD NOTIFIER
# =============================================================================

class DiscordNotifier:
    """Sends alerts to Discord via webhook."""

    def __init__(self, config: Config):
        self.config = config
        self.session = requests.Session()

    def send_alert(self, alert: FreshWhaleAlert) -> bool:
        if not self.config.DISCORD_WEBHOOK_URL:
            return False

        embed = alert.to_discord_embed()
        payload = {"username": "Polymarket Whale Monitor", "embeds": [embed]}

        try:
            response = self.session.post(
                self.config.DISCORD_WEBHOOK_URL,
                json=payload,
                timeout=self.config.REQUEST_TIMEOUT
            )
            return response.status_code == 204
        except requests.exceptions.RequestException:
            return False

    def send_startup_notification(self) -> bool:
        if not self.config.DISCORD_WEBHOOK_URL:
            return False

        payload = {
            "username": "Polymarket Whale Monitor",
            "embeds": [{
                "title": "Whale Monitor Started",
                "description": f"Monitoring for trades above **${self.config.MIN_TRADE_VALUE_USD:,.0f}**\nPolling every **{self.config.POLL_INTERVAL_SECONDS}** seconds",
                "color": 0x0099FF,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "footer": {"text": "Polymarket Whale Monitor"}
            }]
        }

        try:
            response = self.session.post(
                self.config.DISCORD_WEBHOOK_URL,
                json=payload,
                timeout=self.config.REQUEST_TIMEOUT
            )
            return response.status_code == 204
        except requests.exceptions.RequestException:
            return False

# =============================================================================
# WHALE MONITOR
# =============================================================================

class WhaleMonitor:
    """Main monitoring orchestrator."""

    def __init__(self, config: Config):
        self.config = config
        self.client = PolymarketClient(config)
        self.notifier = DiscordNotifier(config)
        self.processed_trade_ids: Set[str] = set()
        self.alerts_history: List[FreshWhaleAlert] = []
        self.stats = {
            "total_trades_scanned": 0,
            "large_trades_found": 0,
            "fresh_whales_detected": 0,
            "alerts_sent": 0,
            "start_time": None
        }

    def process_trade(self, trade: Trade) -> Optional[FreshWhaleAlert]:
        if trade.id in self.processed_trade_ids:
            return None

        self.processed_trade_ids.add(trade.id)
        self.stats["total_trades_scanned"] += 1

        if trade.value_usd < self.config.MIN_TRADE_VALUE_USD:
            return None

        self.stats["large_trades_found"] += 1
        profile = self.client.get_account_profile(trade.user_address)

        if profile is None:
            return None

        if not profile.is_fresh_whale(self.config):
            return None

        if profile.total_trades < self.config.MAX_HISTORICAL_TRADES:
            reason = f"Only {profile.total_trades} prior trades"
        elif profile.account_age_hours is not None and profile.account_age_hours < self.config.NEW_ACCOUNT_HOURS:
            reason = f"Account only {profile.account_age_hours:.1f} hours old"
        else:
            reason = "New account pattern detected"

        self.stats["fresh_whales_detected"] += 1

        alert = FreshWhaleAlert(
            trade=trade,
            profile=profile,
            detection_reason=reason
        )

        self.alerts_history.append(alert)
        return alert

    def run_single_poll(self) -> List[FreshWhaleAlert]:
        alerts = []
        lookback_seconds = self.config.LOOKBACK_MINUTES * 60
        since_timestamp = int(time.time()) - lookback_seconds

        trades = self.client.get_recent_trades(
            since_timestamp=since_timestamp,
            min_value_usd=self.config.MIN_TRADE_VALUE_USD
        )

        for trade in trades:
            alert = self.process_trade(trade)
            if alert:
                alerts.append(alert)
                if self.notifier.send_alert(alert):
                    self.stats["alerts_sent"] += 1

        return alerts

    def display_status(self, iteration: int, alerts: List[FreshWhaleAlert]):
        """Display status in Colab-friendly format."""
        clear_output(wait=True)

        runtime = datetime.now(timezone.utc) - self.stats["start_time"]

        html = f"""
        <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 20px; border-radius: 10px;">
            <h2 style="color: #00ff00;">Polymarket Whale Monitor</h2>
            <hr style="border-color: #00ff00;">
            <p><strong>Status:</strong> Running (Poll #{iteration})</p>
            <p><strong>Runtime:</strong> {runtime}</p>
            <p><strong>Last Check:</strong> {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}</p>
            <hr style="border-color: #00ff00;">
            <h3>Statistics</h3>
            <ul>
                <li>Trades Scanned: {self.stats['total_trades_scanned']}</li>
                <li>Large Trades Found: {self.stats['large_trades_found']}</li>
                <li>Fresh Whales Detected: {self.stats['fresh_whales_detected']}</li>
                <li>Alerts Sent: {self.stats['alerts_sent']}</li>
            </ul>
        """

        if self.alerts_history:
            html += "<hr style='border-color: #ff6600;'><h3 style='color: #ff6600;'>Recent Alerts</h3>"
            for alert in self.alerts_history[-5:]:
                html += f"""
                <div style="background: #2d2d2d; padding: 10px; margin: 10px 0; border-left: 4px solid #ff6600;">
                    <strong style="color: #ff6600;">${alert.trade.value_usd:,.2f}</strong> on 
                    <em>{alert.trade.market_title[:60]}...</em><br>
                    <small>Reason: {alert.detection_reason} | Time: {alert.alert_time.strftime('%H:%M:%S')}</small>
                </div>
                """

        html += "</div>"
        display(HTML(html))

    def run_colab(self, max_iterations: Optional[int] = None):
        """Run monitor with Colab-friendly output."""
        self.stats["start_time"] = datetime.now(timezone.utc)

        print("Starting Polymarket Whale Monitor...")
        print(f"Minimum trade value: ${self.config.MIN_TRADE_VALUE_USD:,}")
        print(f"Poll interval: {self.config.POLL_INTERVAL_SECONDS} seconds")

        self.notifier.send_startup_notification()

        iteration = 0
        try:
            while True:
                iteration += 1

                if max_iterations and iteration > max_iterations:
                    print(f"Reached max iterations ({max_iterations}), stopping")
                    break

                try:
                    alerts = self.run_single_poll()
                    self.display_status(iteration, alerts)
                except Exception as e:
                    print(f"Error in poll cycle: {e}")

                if max_iterations is None or iteration < max_iterations:
                    time.sleep(self.config.POLL_INTERVAL_SECONDS)

        except KeyboardInterrupt:
            print("\nMonitor stopped by user.")

print("Core monitoring code loaded successfully!")

## Step 4: Test the Connection

Let's verify the API connections are working before starting the monitor.

In [ ]:
# Test the API connections
test_config = Config()
test_client = PolymarketClient(test_config)

print("Testing Polymarket API connections...")
print("="*50)

# Test Activity Subgraph
print("\n1. Testing Activity Subgraph...")
since_ts = int(time.time()) - 3600  # Last hour
min_amount = int(1000 * 1_000_000)  # $1000 for testing
splits = test_client.get_large_splits(since_ts, min_amount, first=5)
if splits:
    print(f"   SUCCESS! Found {len(splits)} recent splits")
    s = splits[0]
    print(f"   Sample: {s['stakeholder'][:12]}... - ${int(s['amount'])/1_000_000:,.2f}")
else:
    print("   No splits found in the last hour above $1000")

# Test Gamma API
print("\n2. Testing Gamma API...")
try:
    response = requests.get(f"{test_config.GAMMA_API_URL}/markets", params={"limit": 1}, timeout=10)
    if response.status_code == 200:
        markets = response.json()
        if markets:
            print(f"   SUCCESS! Sample market: {markets[0].get('question', 'N/A')[:50]}...")
        else:
            print("   Connected but no markets returned")
    else:
        print(f"   WARNING: Status {response.status_code}")
except Exception as e:
    print(f"   ERROR: {e}")

# Test Data API
print("\n3. Testing Data API...")
try:
    # Use a sample address that likely exists
    if splits:
        sample_addr = splits[0]['stakeholder']
        activities = test_client.get_user_activity(sample_addr, limit=5)
        if activities:
            print(f"   SUCCESS! Found {len(activities)} activities for sample user")
        else:
            print("   Connected but no activities returned")
    else:
        print("   Skipped (no sample address available)")
except Exception as e:
    print(f"   ERROR: {e}")

print("\n" + "="*50)
print("API connection test complete!")

## Step 5: Start the Whale Monitor

Run the cell below to start monitoring. The display will update every polling cycle.

**To stop the monitor:** Click the "Stop" button in the Colab toolbar, or press `Ctrl+M I` to interrupt.

---

In [ ]:
# =============================================================================
# START THE WHALE MONITOR
# =============================================================================

# Build configuration from the settings above
config = Config(
    DISCORD_WEBHOOK_URL=DISCORD_WEBHOOK_URL,
    MIN_TRADE_VALUE_USD=MIN_TRADE_VALUE_USD,
    MAX_HISTORICAL_TRADES=MAX_HISTORICAL_TRADES,
    NEW_ACCOUNT_HOURS=NEW_ACCOUNT_HOURS,
    POLL_INTERVAL_SECONDS=POLL_INTERVAL_SECONDS,
    LOOKBACK_MINUTES=LOOKBACK_MINUTES
)

# Create and run the monitor
monitor = WhaleMonitor(config)

# Run indefinitely (or set max_iterations for testing)
# For testing, uncomment the line below:
# monitor.run_colab(max_iterations=5)

# For production, run indefinitely:
monitor.run_colab()

---

## Viewing Alert History

After stopping the monitor, run this cell to see all detected whales:

In [ ]:
# View all detected Fresh Whales
if 'monitor' in dir() and monitor.alerts_history:
    df = pd.DataFrame([
        {
            "Time": alert.alert_time.strftime("%Y-%m-%d %H:%M:%S"),
            "Value (USD)": f"${alert.trade.value_usd:,.2f}",
            "Market": alert.trade.market_title[:50] + "...",
            "Outcome": alert.trade.outcome,
            "Price": f"${alert.trade.price:.3f}",
            "Account Age (hrs)": f"{alert.profile.account_age_hours:.1f}" if alert.profile.account_age_hours else "N/A",
            "Prior Trades": alert.profile.total_trades,
            "Reason": alert.detection_reason,
            "Wallet": f"{alert.trade.user_address[:8]}...{alert.trade.user_address[-6:]}"
        }
        for alert in monitor.alerts_history
    ])
    display(df)
else:
    print("No alerts detected yet. Run the monitor first!")

---

## Limitations: Google Colab vs VPS

### Google Colab Limitations:

1. **Session Timeouts**: Colab disconnects after ~90 minutes of inactivity, and free tier has ~12-hour maximum runtime
2. **No Background Execution**: When the browser tab is closed, execution stops
3. **Resource Limits**: Free tier has limited RAM and compute
4. **IP Rotation**: Colab IP addresses change, which can cause issues with rate limiting

### VPS Advantages:

1. **24/7 Uptime**: Runs continuously without interruption
2. **Persistent State**: Can store historical data and resume after restarts
3. **Scheduled Tasks**: Use cron or systemd for automatic restarts
4. **More Resources**: Dedicated RAM, CPU, and storage
5. **Static IP**: Consistent IP for API rate limiting

### Migrating to VPS:

1. Copy `whale_monitor.py` to your VPS
2. Install dependencies: `pip install -r requirements.txt`
3. Set environment variable: `export DISCORD_WEBHOOK_URL="your_url"`
4. Run: `python whale_monitor.py`
5. For persistence, use: `nohup python whale_monitor.py &` or create a systemd service

---